# Week 61, Day 5: UMAP, and reading embeddings without being fooled

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w061/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sqdist(A):
    s = (A ** 2).sum(1)
    return np.maximum(s[:, None] + s[None] - 2 * A @ A.T, 0)

def joint_p(X, perp):
    D2 = sqdist(X); n = len(X); P = np.zeros((n, n)); target = np.log(perp)
    for i in range(n):
        d = np.delete(D2[i], i); lo, hi, beta = 0.0, np.inf, 1.0 / np.median(d)
        for _ in range(60):
            p = np.exp(-(d - d.min()) * beta); p /= p.sum()
            H = -(p * np.log(np.maximum(p, 1e-300))).sum()
            if abs(H - target) < 1e-4: break
            if H > target: lo = beta; beta = beta * 2 if hi == np.inf else (beta + hi) / 2
            else: hi = beta; beta = (beta + lo) / 2
        P[i, np.arange(n) != i] = p
    return (P + P.T) / (2 * n)

def tsne(X, perp=30, iters=500, seed=0, init=None, exaggeration=12.0):
    P = joint_p(X, perp)
    g = np.random.default_rng(seed)
    Y = 1e-4 * g.normal(size=(len(X), 2)) if init is None else init.copy()
    V = np.zeros_like(Y)
    for it in range(iters):
        exag, mom = (exaggeration, 0.5) if it < 100 else (1.0, 0.8)
        W = 1 / (1 + sqdist(Y)); np.fill_diagonal(W, 0); Q = W / W.sum()
        M = (exag * P - Q) * W
        V = mom * V - 100 * 4 * (M.sum(1)[:, None] * Y - M @ Y)
        Y = Y + V
    return Y

def silhouette(Z, lab):
    D = np.sqrt(sqdist(Z)); s = []
    for i in range(len(Z)):
        same = lab == lab[i]; same[i] = False
        a = D[i, same].mean()
        b = min(D[i, lab == c].mean() for c in np.unique(lab) if c != lab[i])
        s.append((b - a) / max(a, b))
    return np.mean(s)
print("ready: tsne(X, perp, iters, seed, init) and silhouette(Z, labels)")

### Exercise: A ring of head-direction activity

Simulate 200 population vectors of 40 head-direction cells for angles evenly spaced round the circle. Embed them with t-SNE at perplexity 5 and project them with PCA. For each, compute the largest step between consecutive angles divided by the median step.

In [ ]:
gr = np.random.default_rng(615)
ang = np.linspace(0, 2 * np.pi, 200, endpoint=False)
pref = np.linspace(0, 2 * np.pi, 40, endpoint=False)
R = np.exp(2 * (np.cos(ang[:, None] - pref[None]) - 1)) + 0.15 * gr.normal(size=(200, 40))

def max_jump(Z):
    steps = np.linalg.norm(np.roll(Z, -1, axis=0) - Z, axis=1)
    return steps.max() / np.median(steps)

Y_ring = tsne(R, perp=5, seed=1)
Rc = R - R.mean(0)
Y_ring_pca = Rc @ np.linalg.svd(Rc, full_matrices=False)[2][:2].T
jump_data, jump_pca = max_jump(R), max_jump(Y_ring_pca)
jump_tsne = ...
print(f"largest step / median step: data {jump_data:.1f}, PCA {jump_pca:.1f}, t-SNE (perplexity 5) {jump_tsne:.1f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].scatter(*Y_ring_pca.T, c=ang, cmap="hsv", s=8); ax[0].set_title("PCA")
ax[1].scatter(*Y_ring.T, c=ang, cmap="hsv", s=8); ax[1].set_title("t-SNE, perplexity 5"); plt.show()

In [ ]:
import numpy as np
def test_breaks():
    "at perplexity 5, the embedding has a jump between consecutive angles more than 3 times larger, relative to the median step, than the data or the PCA projection"
    assert jump_tsne > 3 * max(jump_data, jump_pca)

test_breaks()
print("All checks passed")

### Exercise: Clumps in a Gaussian cloud

Embed 250 points of 30-dimensional Gaussian noise with t-SNE at perplexity 5. Cluster both the embedding and the original data into 6 groups with k-means, and compare their silhouette scores (higher means better separated).

In [ ]:
from scipy.cluster.vq import kmeans2
gz = np.random.default_rng(6151)
Xn = gz.normal(size=(250, 30))
Y_noise = tsne(Xn, perp=5, seed=2)
_, lab_emb = kmeans2(Y_noise, 6, minit="++", seed=3)
_, lab_dat = kmeans2(Xn, 6, minit="++", seed=3)
sil_emb = silhouette(Y_noise, lab_emb)
sil_data = ...
print(f"silhouette of 6 k-means clusters: in the embedding {sil_emb:.3f}, in the data {sil_data:.3f}")
plt.scatter(*Y_noise.T, c=lab_emb, cmap="tab10", s=8); plt.title("t-SNE of pure noise, perplexity 5"); plt.show()

In [ ]:
import numpy as np
def test_clumps():
    "k-means clusters look much better separated in the embedding of pure noise than in the noise itself"
    assert sil_emb > sil_data + 0.15

test_clumps()
print("All checks passed")

### Exercise: Random against PCA initialisation

Simulate 8 clusters whose centres lie along a curve (so there is a global order), embed them with t-SNE without early exaggeration from 4 random starts and from a PCA start (the first two PC scores scaled to standard deviation 1e-4), and for each embedding compute the correlation between the 28 pairwise distances of cluster centres in the embedding and in the data.

In [ ]:
gi = np.random.default_rng(6152)
s_ = np.linspace(0, 1, 8)
cent = np.column_stack([np.cos(3 * s_), np.sin(3 * s_), s_] + [0 * s_] * 17) * 12
lab8 = np.repeat(np.arange(8), 30)
X8 = cent[lab8] + gi.normal(size=(240, 20))

def centre_corr(Z):
    C = np.array([Z[lab8 == c].mean(0) for c in range(8)])
    Cx = np.array([X8[lab8 == c].mean(0) for c in range(8)])
    iu = np.triu_indices(8, 1)
    return np.corrcoef(np.sqrt(sqdist(C))[iu], np.sqrt(sqdist(Cx))[iu])[0, 1]

corr_rand = [centre_corr(tsne(X8, 30, seed=s, exaggeration=1.0)) for s in range(4)]
Xc8 = X8 - X8.mean(0)
pcs = Xc8 @ np.linalg.svd(Xc8, full_matrices=False)[2][:2].T
init_pca = ...
corr_pca = centre_corr(tsne(X8, 30, init=init_pca, exaggeration=1.0))
print("random starts:", np.round(corr_rand, 3), f"PCA start: {corr_pca:.3f}")

In [ ]:
import numpy as np
def test_init():
    "with PCA initialisation, distances between cluster centres in the embedding track those in the data better than the average random start"
    assert corr_pca > np.mean(corr_rand) + 0.1

test_init()
print("All checks passed")